In [17]:
csv_path_r = find_file("resolution_time_regression.csv")
df_r = pd.read_csv(csv_path_r)

print("Loaded from:", csv_path_r)
print("Shape:", df_r.shape)
print("Columns:", list(df_r.columns))
print("\nTarget summary ('resolution_time_hours'):")
print(df_r["resolution_time_hours"].describe())
df_r.head()

Loaded from: resolution_time_regression[1].csv
Shape: (800, 7)
Columns: ['ticket_id', 'complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months', 'resolution_time_hours']

Target summary ('resolution_time_hours'):
count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [18]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

target_col_r = "resolution_time_hours"
drop_cols_r = [c for c in ["ticket_id", target_col_r] if c in df_r.columns]

X_r = df_r.drop(columns=drop_cols_r)
y_r = df_r[target_col_r].astype(float)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_r,
    y_r,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("X_train_r shape:", X_train_r.shape)
print("X_test_r shape: ", X_test_r.shape)

X_train_r shape: (640, 5)
X_test_r shape:  (160, 5)


In [19]:
target_col_c = "escalated"
drop_cols_c = [c for c in ["ticket_id", target_col_c] if c in df_c.columns]

X_c = df_c.drop(columns=drop_cols_c)
y_c = df_c[target_col_c].astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c,
    y_c,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_c,
)

print("X_train_c shape:", X_train_c.shape, "| Escalated rate:", y_train_c.mean().round(4))
print("X_test_c shape: ", X_test_c.shape,  "| Escalated rate:", y_test_c.mean().round(4))

X_train_c shape: (720, 5) | Escalated rate: 0.2069
X_test_c shape:  (180, 5) | Escalated rate: 0.2056


In [20]:
pipe_ridge_r = Pipeline([
    ("prep", preprocessor_r),
    ("model", Ridge(random_state=RANDOM_STATE)),
])
param_grid_ridge_r = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
}
grid_ridge_r = GridSearchCV(
    pipe_ridge_r,
    param_grid_ridge_r,
    cv=cv_r,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_ridge_r.fit(X_train_r, y_train_r)

pipe_dt_r = Pipeline([
    ("prep", preprocessor_r),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
])
param_grid_dt_r = {
    "model__max_depth": [2, 3, 4, 5, 6, 8, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
}
grid_dt_r = GridSearchCV(
    pipe_dt_r,
    param_grid_dt_r,
    cv=cv_r,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_dt_r.fit(X_train_r, y_train_r)

print(f"Best Ridge:                 {grid_ridge_r.best_params_} | CV MAE: {-grid_ridge_r.best_score_:.4f}")
print(f"Best DecisionTreeRegressor: {grid_dt_r.best_params_} | CV MAE: {-grid_dt_r.best_score_:.4f}")

best_search_r = grid_ridge_r if grid_ridge_r.best_score_ >= grid_dt_r.best_score_ else grid_dt_r
best_model_r = best_search_r.best_estimator_

print("\nSelected best estimator:", best_model_r.named_steps["model"].__class__.__name__, best_search_r.best_params_)

Best Ridge:                 {'model__alpha': 1.0} | CV MAE: 2.0361
Best DecisionTreeRegressor: {'model__max_depth': 6, 'model__min_samples_leaf': 20} | CV MAE: 2.3836

Selected best estimator: Ridge {'model__alpha': 1.0}


In [23]:
y_pred_r = best_model_r.predict(X_test_r)

test_mae_r = mean_absolute_error(y_test_r, y_pred_r)
test_rmse_r = root_mean_squared_error(y_test_r, y_pred_r)

print(f"Test MAE:  {test_mae_r:.4f} hours")
print(f"Test RMSE: {test_rmse_r:.4f} hours")

Test MAE:  2.2270 hours
Test RMSE: 3.3988 hours


## 4. Experiment B — Regression Results, Outliers & MAE vs. RMSE

### Validation Strategy, Candidate Models & Search Space
* **Validation strategy:** **5-fold Cross-Validation (`KFold`)** on the `640`-row training set (`160` rows held out for the final test set).
* **Candidate models & tuning search space:** Compared `LinearRegression`, `Ridge`, `KNNRegressor`, and `DecisionTreeRegressor` using 5-fold CV with `MAE` (`neg_mean_absolute_error`) and `RMSE` (`neg_root_mean_squared_error`). Then tuned `Ridge` (`alpha` in `[0.01, 0.1, 1.0, 10.0, 100.0]`) and `DecisionTreeRegressor` (`max_depth` and `min_samples_leaf` grids) with `GridSearchCV` using negative MAE.

### 5-Fold CV & Hyperparameter Tuning Results
* **Best tuned `Ridge`:** `{'model__alpha': 1.0}` with **CV MAE = `2.0361` hours**.
* **Best tuned `DecisionTreeRegressor`:** `{'model__max_depth': 6, 'model__min_samples_leaf': 20}` with **CV MAE = `2.3836` hours**.
* **Selected best estimator:** `Ridge` (`alpha=1.0`), selected because its CV MAE (`2.0361` hours) was lower (better) than `DecisionTreeRegressor` (`2.3836` hours).

### One-Time Held-Out Test Set Evaluation (`160` tickets)
* **Test MAE:** `2.2270` hours (on average, predictions are off by about `2.23` hours).
* **Test RMSE:** `3.3988` hours.

### Outliers and Why MAE and RMSE May Prefer Different Models
In `resolution_time_hours`, most tickets cluster around the median (`5.3874` hours) and 75th percentile (`8.3459` hours), while the maximum reaches `27.4882` hours—creating right-skewed high-resolution outliers (tickets that take much longer than usual).

* **MAE** averages absolute errors (`|y_true - y_pred|`), giving equal weight per hour of error. It tends to favor models that predict typical/central resolution times accurately across the majority of tickets.
* **RMSE** squares errors before averaging and taking the square root, so large errors (such as missing a `27.49`-hour outlier ticket) are penalized much more heavily than small errors. Because squaring magnifies big misses, **RMSE** can prefer a model that reduces extreme errors even if another model achieves a slightly lower **MAE** overall.

### Final Model Justification & Limitations
* **Justification:** `Ridge` (`alpha=1.0`) achieved the lowest 5-fold CV MAE (`2.0361` hours) among the tuned candidates and generalized with a reasonable test MAE (`2.2270` hours) and test RMSE (`3.3988` hours) on the `160`-row test set.
* **Limitations:** Test MAE (`2.2270` hours) is slightly higher than CV MAE (`2.0361` hours), and Test RMSE (`3.3988` hours) is notably higher than Test MAE (`2.2270` hours), reflecting larger errors on longer-running or outlying tickets and potential non-linear patterns that a linear `Ridge` model cannot capture.

## 5. Reflection Questions

1. **Why stratify the classification split?**
   Stratification preserves the class proportion (`escalated = 0` vs. `1`, ~20.7% positive) across the train split, test split, and CV folds so no split or fold ends up unrepresentative or missing minority-class examples.

2. **Why must preprocessing be inside the pipeline?**
   Placing imputation, scaling, and one-hot encoding inside the `Pipeline` ensures those steps are fit only on the training data/folds and then applied to validation and test data, preventing data leakage and keeping inference consistent.

3. **What does fold variability tell you?**
   Fold variability (such as `CV F1 Std` of `0.0453` to `0.0604` across the 5 folds in Experiment A) shows how much a model's score fluctuates across different subsets of the training data, indicating stability or sensitivity to sample composition and noise.

4. **Why should the test set be evaluated only once?**
   Evaluating the held-out test set only once after model selection gives an unbiased estimate of real-world generalization; repeatedly checking test scores and adjusting choices based on them leaks test information and inflates confidence in the results.

5. **What deployment constraints might change your choice?**
   Production constraints such as prediction latency/throughput limits, memory footprint, interpretability/audit requirements (for example, explaining why a ticket was flagged or given a time estimate), or differing costs of false positives vs. false negatives could lead to choosing a simpler, faster, or more constrained model even if another model scored slightly higher on a single validation metric.